## 1. 对比标准化后的准确率高于标准化前

In [1]:
# 导入必要的库
import matplotlib as mpl
import matplotlib.pyplot as plt
# 在Jupyter notebook中内联显示图表
%matplotlib inline  
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm  # 进度条库
import torch
import torch.nn as nn
import torch.nn.functional as F

# 打印Python版本信息
print(sys.version_info)

# 打印各个库的版本信息
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
# 设置设备：如果有GPU则使用GPU，否则使用CPU
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.10.8
numpy 2.4.0
pandas 2.3.3
sklearn 1.8.0
torch 2.9.1+cpu
cpu


In [ ]:
# 导入torchvision的datasets和transforms模块
# datasets用于包含多种常用的数据集（如FashionMNIST），transforms用于对图片进行一系列预处理操作
from torchvision import datasets, transforms  # datasets用于加载数据集，transforms用于对图像数据做预处理

# 定义图片数据的预处理流程
# transforms.Compose可以将多个数据变换（transforms）操作组合成一个整体，按顺序对输入数据执行所有变换
transform = transforms.Compose([
    transforms.ToTensor(),  
    # ToTensor()将PIL图像或者 numpy.ndarray 格式的数据转换为 torch.FloatTensor，并将像素值归一化到[0,1]之间
    transforms.Normalize((0.2860,), (0.3205,))
    # Normalize标准化操作，对每个通道执行：(像素值-均值)/标准差。这里均值为0.2860，标准差为0.3205
    # 这些值需要提前通过训练集像素统计得到，使得标准化后数据均值为0，方差为1
])

# 加载训练集
train_dataset = datasets.FashionMNIST(
    root='./data',       # 数据将保存或者查找在'./chapter2/data'这个本地文件夹下
    train=True,                   # train=True表示获取的是训练集，总共60000张图像
    download=True,                # 如果数据本地没有则自动从网络下载
    transform=transform           # 应用上面定义的transform，对每张图片进行ToTensor和Normalize预处理
)

# 加载测试集，一般在模型评估时使用
test_dataset = datasets.FashionMNIST(
    root='./data',       # 保存和训练集相同的目录下
    train=False,                  # train=False表示获取的是测试集，总共10000张图像
    download=True,                # 如果本地没有也同样会自动下载
    transform=transform           # 对测试集图片做和训练集一致的预处理（保证两者分布一致）
)

# 打印输出训练集和测试集的样本数量，便于确认数据是否加载正确
print(f"训练集大小: {len(train_dataset)}")   # FashionMNIST训练集应有60000张图像
print(f"测试集大小: {len(test_dataset)}")   # 测试集应有10000张图像

# FashionMNIST数据集包含10个服饰类别，每个类别对应一个整数标签，标签名字如下
# 类别标签说明可参考项目页面：https://github.com/zalandoresearch/fashion-mnist
class_names = ['T-shirt/top',    # 0: T恤/上衣
               'Trouser',        # 1: 裤子
               'Pullover',       # 2: 套头衫
               'Dress',          # 3: 连衣裙
               'Coat',           # 4: 外套
               'Sandal',         # 5: 凉鞋
               'Shirt',          # 6: 衬衫
               'Sneaker',        # 7: 运动鞋
               'Bag',            # 8: 包
               'Ankle boot']     # 9: 靴子

# 输出类别数量和详细标签名，便于后续解读模型输出
print(f"类别数量: {len(class_names)}")      # 输出类别总数，应该为10
print(f"类别标签: {class_names}")           # 输出类别标签的具体名称列表


 41%|████      | 10.8M/26.4M [00:29<00:41, 373kB/s] 


KeyboardInterrupt: 

In [ ]:
import torch  # 导入PyTorch库

# -------------------- 计算整个训练集的均值和标准差（详解注释） ------------------------
# 构建DataLoader，每批次加载5000张图片。shuffle=False保持图片顺序，num_workers=0表示使用主进程加载数据。
loader = torch.utils.data.DataLoader(
    train_dataset,           # 使用已经加载好的FashionMNIST训练集
    batch_size=5000,         # 每个小批量加载5000张图片，提高均值/方差统计效率
    shuffle=False,           # 按原始顺序读取数据，便于复现实验结果
    num_workers=0            # 数据加载worker数量，0为主进程
)

n_samples = 0          # 用于统计已处理过的图片总数，初始为0
mean = 0.0             # 用于累加所有图片的均值
std = 0.0              # 用于累加所有图片的标准差

# 遍历数据加载器，逐批取出图片和标签。这里只关心图片(images)，忽略标签(_)
for images, _ in loader:
    # images的形状为[batch_size, 1, 28, 28]
    batch_samples = images.size(0)          # 当前批次的图片数量（最后一批可能小于5000）
    # 将每张图片展平成一维，变成[batch_size, 784]，方便后续均值/标准差运算
    images = images.view(batch_samples, -1) 
    # 对每张图片（共batch_samples张）分别计算均值，得到长度为batch_samples的均值向量后累加
    mean += images.mean(1).sum()            # images.mean(1)是对每行求均值（即每张图片），再.sum()加总本batch所有均值
    # 对每张图片分别计算标准差，累加本批次所有图片的标准差
    std += images.std(1).sum()              # images.std(1)是对每行求标准差（即每张图片），再.sum()加总本batch所有标准差
    n_samples += batch_samples              # 批次图片数量累加进总样本数

# 所有batch遍历完成后，mean和std分别是全体图片均值和标准差（“总和”），需要除以图片总数得到平均值
mean /= n_samples  # 得到所有图片均值的平均（最终整体均值）
std /= n_samples   # 得到所有图片标准差的平均（最终整体标准差）

# 打印最终的训练集均值与标准差，便于后续设置Normalize参数
print(f"训练集均值: {mean.item():.4f}")      # 打印整体像素均值
print(f"训练集标准差: {std.item():.4f}")     # 打印整体像素标准差


训练集均值: 0.0001
训练集标准差: 0.9999


In [ ]:
from torch.utils.data import DataLoader

# 创建训练集和验证集的DataLoader
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # 训练时打乱数据
    num_workers=2  # 使用多进程加载数据
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,  # 测试时不需要打乱
    num_workers=2
)

In [ ]:
import torch.nn as nn

class TwoLayerNN(nn.Module):
    def __init__(self):
        super(TwoLayerNN, self).__init__()
        self.flatten = nn.Flatten() # 展平操作，将28*28的图像展平为784维的向量
        self.fc1 = nn.Linear(28*28, 300) # 全连接层，将784维的向量映射为300维的向量
        self.relu1 = nn.ReLU() # 激活函数，ReLU函数
        self.fc2 = nn.Linear(300, 100) # 全连接层，将300维的向量映射为100维的向量
        self.relu2 = nn.ReLU() # 激活函数，ReLU函数
        self.fc3 = nn.Linear(100, 10) # 全连接层，将100维的向量映射为10维的向量
    # 在神经网络中：
    # - 输入层负责接收原始数据（如28*28的图片像素，共784个特征），对应于self.flatten输出的向量。
    # - 隐藏层是一或多个“中间”全连接层（如self.fc1和self.fc2），用于提取和变换特征，通常跟随激活函数（如ReLU）。
    # - 输出层（如self.fc3）将隐藏层输出映射为最终结果（如10类的概率分布）。
    # 全连接层（nn.Linear）可用于输入层到隐藏层、隐藏层之间、以及最后的输出层。
        
    def forward(self, x):
        # 先将输入x（形状为batch_size, 1, 28, 28）的图像展平成一维向量（batch_size, 784）
        x = self.flatten(x)
        # 通过第一个全连接层，将784维向量映射为300维
        x = self.fc1(x)
        # 使用ReLU激活函数，增加非线性，帮助模型学习更复杂的特征
        x = self.relu1(x)
        # 通过第二个全连接层，将300维向量映射为100维
        x = self.fc2(x)
        # 再次使用ReLU激活
        x = self.relu2(x)
        # 最后通过输出层，将100维向量映射为10维（表示10个类别的分数）
        x = self.fc3(x)

        return x

model_normalize = TwoLayerNN()
print(model_normalize)


TwoLayerNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=300, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=300, out_features=100, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=100, out_features=10, bias=True)
)


In [ ]:
# state_dict()方法返回当前模型的参数字典（权重和偏置等），类型为collections.OrderedDict
print(type(model_normalize.state_dict()))  # 输出state_dict的类型，通常为OrderedDict

# 查看模型的参数字典，其中存储着模型的所有可学习参数（例如不同层的权重和偏置）
# 这个字典通常在模型保存（torch.save）或加载（load_state_dict）时用到
model_state = model_normalize.state_dict()  # 获取模型参数字典
print(model_state)  # 打印参数字典以便查看每一层的参数信息

<class 'collections.OrderedDict'>


OrderedDict([('fc1.weight',
              tensor([[-0.0129, -0.0068, -0.0023,  ...,  0.0116, -0.0058, -0.0073],
                      [ 0.0017,  0.0321,  0.0127,  ...,  0.0163,  0.0293, -0.0298],
                      [ 0.0166,  0.0216,  0.0173,  ...,  0.0226, -0.0154, -0.0218],
                      ...,
                      [ 0.0343,  0.0031, -0.0029,  ..., -0.0293,  0.0227,  0.0148],
                      [-0.0333,  0.0301,  0.0260,  ..., -0.0247, -0.0234, -0.0079],
                      [-0.0337, -0.0193, -0.0016,  ...,  0.0187,  0.0055,  0.0025]])),
             ('fc1.bias',
              tensor([ 2.3926e-02, -1.1940e-02, -2.3268e-02,  2.1642e-02, -2.4472e-02,
                      -1.8960e-02,  2.5509e-03, -2.3211e-02,  3.2643e-02,  2.8945e-02,
                      -1.3740e-02,  7.0576e-03, -1.8985e-02,  1.7066e-02, -2.6402e-02,
                       1.3694e-02,  1.6439e-02, -5.1800e-03,  1.7890e-02,  1.7067e-02,
                       5.6442e-03, -2.0455e-02, -1.5251e-02,  2.

In [ ]:
import torch.nn as nn
import torch.optim as optim

# 初始化交叉熵损失函数，内部会做softmax
criterion = nn.CrossEntropyLoss()

# 初始化优化器（这里选用Adam，也可以使用SGD等）
optimizer_normalize = optim.Adam(model_normalize.parameters(), lr=0.001)

In [ ]:
import chapter2.wangdao_train as wangdao_train  # 导入自定义的训练工具（Trainer类）

# 假设train_loader为训练集的数据加载器，test_loader为测试集的数据加载器
# device为模型训练所用设备，通常为"cuda"（GPU）或"cpu"（CPU）

# 创建Trainer实例，用于封装训练和验证逻辑
trainer_normalize = wangdao_train.Trainer(
    model=model_normalize,           # 需要训练的模型（此处为TwoLayerNN的实例）
    train_loader=train_loader,       # 提供训练数据的数据加载器
    val_loader=test_loader,          # 提供验证（测试）数据的数据加载器
    criterion=criterion,             # 损失函数，这里采用交叉熵损失
    optimizer=optimizer_normalize,   # 优化器，这里采用Adam
    device=device                    # 指定训练所用设备
)

# 设定训练轮数，这里设为10轮
num_epochs = 10

# 调用Trainer的train方法，开始模型的训练过程
# train方法内部会循环num_epochs轮，并进行训练与验证（可输出loss, accuracy等）
trainer_normalize.train(num_epochs)


[Step 100] Val Loss: 1.3356 Val Acc: 0.6549
[Step 200] Val Loss: 1.2683 Val Acc: 0.6676
[Step 300] Val Loss: 1.2100 Val Acc: 0.6667
[Step 400] Val Loss: 1.1592 Val Acc: 0.6760
[Step 500] Val Loss: 1.1159 Val Acc: 0.6820
[Step 600] Val Loss: 1.0763 Val Acc: 0.6937
[Step 700] Val Loss: 1.0413 Val Acc: 0.7012
[Step 800] Val Loss: 1.0110 Val Acc: 0.7055
[Step 900] Val Loss: 0.9820 Val Acc: 0.7071
[Step 1000] Val Loss: 0.9575 Val Acc: 0.7139
[Step 1100] Val Loss: 0.9333 Val Acc: 0.7161
[Step 1200] Val Loss: 0.9114 Val Acc: 0.7213
[Step 1300] Val Loss: 0.8931 Val Acc: 0.7211
[Step 1400] Val Loss: 0.8748 Val Acc: 0.7258
[Step 1500] Val Loss: 0.8592 Val Acc: 0.7272
[Step 1600] Val Loss: 0.8423 Val Acc: 0.7329
[Step 1700] Val Loss: 0.8283 Val Acc: 0.7303
[Step 1800] Val Loss: 0.8154 Val Acc: 0.7303
Epoch [1/10]  Train Loss: 1.0060  Train Acc: 0.7080
[Step 1900] Val Loss: 0.8025 Val Acc: 0.7382
[Step 2000] Val Loss: 0.7897 Val Acc: 0.7409
[Step 2100] Val Loss: 0.7791 Val Acc: 0.7406
[Step 2200] 

In [ ]:
trainer_normalize.plot_curves()

## 2. 完成早停，模型保存实战

In [ ]:
import chapter2.wangdao_train as wangdao_train  # 导入自定义的训练辅助模块
# 为了确保编辑器/交互环境中可立即看到代码更新，强制重新加载模块
import importlib
importlib.reload(wangdao_train)

# --------------------------
# 1. 配置早停（EarlyStopping）
# --------------------------
# 作用：当模型在验证集（validation set）上的评价指标（如准确率）一段时间没有提升时，自动中止训练，防止过拟合
# 参数说明:
#   patience=5      ：验证指标超过5个eval_step没有提升就停下来
#   min_delta=0.01  ：只有提升大于0.01才认为是“有提升”，否则视为未提升
#   mode='max'      ：'max' 表示监控的指标是“越大越好”（如准确率），'min'常用于损失(loss)
early_stopping = wangdao_train.EarlyStopping(
    patience=5,
    min_delta=0.01,
    mode='max'
)

# --------------------------
# 2. 配置模型保存（ModelCheckpoint）
# --------------------------
# 作用：在训练过程中根据验证集表现自动保存模型快照
# 参数说明:
#   filepath             ：文件保存模板，{epoch}会被实际轮数替换
#   monitor='val_acc'    ：以验证准确率为判断标准
#   save_best_only=True  ：只保存“最优模型”而非每一次（节省空间）
#   mode='max'           ：监控参数越高越好，'min'常用于损失
#   min_delta=0.01       ：至少提升0.01才判定为“更优”
model_checkpoint = wangdao_train.ModelCheckpoint(
    filepath='./checkpoints/fashion_model_epoch_{epoch}.pt',
    monitor='val_acc',
    save_best_only=True,
    mode='max',
    min_delta=0.01
)

# --------------------------
# 3. 配置TensorBoard回调
# --------------------------
# 作用：用于可视化训练过程（loss、准确率、学习率变化曲线等）
# 参数说明:
#   log_dir ：日志文件保存路径，会被TensorBoard读取用于显示曲线图等
tensorboard_callback = wangdao_train.TensorBoardCallback(log_dir='./logs')

# --------------------------
# 4. 配置训练环境与Trainer对象
# --------------------------
# 设备自动检测，如果可用则用GPU，否则用CPU
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

# 定义Trainer实例，用于集中管理训练、验证、早停、保存和可视化等逻辑
# 主要参数说明如下
#   model           ：需要训练的神经网络模型
#   train_loader    ：训练数据读取器
#   val_loader      ：验证（测试）数据读取器
#   criterion       ：损失函数（如交叉熵）
#   optimizer       ：优化器（如Adam）
#   device          ：计算设备
#   eval_step       ：每训练多少个batch就进行一次验证
#   early_stopping  ：早停对象（提前中止训练）
#   model_checkpoint：模型保存对象
#   tensorboard_callback：TensorBoard记录对象
trainer_normalize = wangdao_train.Trainer(
    model=model_normalize,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer_normalize,
    device=device,
    eval_step=500,                   # 每训练500步在验证集评估并可触发保存/早停
    early_stopping=early_stopping,
    model_checkpoint=model_checkpoint,
    tensorboard_callback=tensorboard_callback
)

# --------------------------
# 5. 开始训练
# --------------------------
num_epochs = 100  # 最大训练轮数

trainer_normalize.train(num_epochs)  # 执行训练过程，内部自动处理验证、早停、模型保存与记录可视化
